# DeepRL Monopoly — self-play vs Builder + DealMaker + 1x ASU

**THIS IS THE CONFIRMED REAL TARGET TABLE** (1 ASU seat + 2 fixed agents,
not 3x ASU, not 1v1). Matches the `asu_mixed` reference logs where
CHAMPION.pt scored 25.9% over 27 games.

Pure self-play RL — ASU is called strictly as a black-box opponent via
choose_action(env). No imitation loss, no reading ASU outputs as labels,
no distillation. This table is slower than fixed-only tables (1 ASU
decision per round adds real latency) but much faster than 3x ASU.

## 1. Mount Drive (own checkpoint folder)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepRL_Monopoly_ckpt_target'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Clone the repo (feature/asu-teacher-distillation branch)

In [ ]:
%cd /content
!rm -rf DeepRL_Monopoly
!git clone --branch feature/asu-teacher-distillation https://github.com/EnzeCbe/monopoly-boom.git DeepRL_Monopoly
%cd DeepRL_Monopoly

## 3. Check GPU + torch

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 4. Train vs Builder + DealMaker + ASU

In [ ]:
import os
OUT = f"{CHECKPOINT_DIR}/vs_target_1asu2fixed.pt"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

# Warm-started from our own proxy-table checkpoint (9000+ games vs
# Builder+DealMaker+Hoarder) instead of random weights — real skill
# transfer, not ASU-related. See tools/train_vs_target_table.py --warm-start.
!PYTHONIOENCODING=utf-8 python tools/train_vs_target_table.py \
  --algo ppo --games 3000 --seed 55 \
  --warm-start artifacts/warm_start/proxy_table_checkpoint.pt \
  --checkpoint-every 25 --log-every 15 \
  --out "{OUT}"

## 5. Resume after a disconnect

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/train_vs_target_table.py \
  --algo ppo --games 20000 --seed 55 --resume \
  --checkpoint-every 25 --log-every 15 \
  --out "{OUT}"

## 6. Analyze the per-game log

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/analyze_run.py "{OUT.rsplit('.', 1)[0]}_games.csv" --window 50